# Week 6 — Monday: Classification with k-Nearest Neighbors

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/06/images/penguins_art.png" width="380" alt="Drawings of the three penguin species: a Chinstrap, a Gentoo and an Adelie. Artwork by @allison_horst.">

Penguins measured by a field team at Palmer Station, Antarctica — and this time each one comes with an **answer**: its species.

> 🎯 **A new penguin's bill is 44.1 mm long and 18.0 mm deep. Which species is it?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Look at the data |
| ~10 min | 2 · Ask the neighbors: k-nearest neighbors (SLO 06A) |
| ~7 min | 3 · New penguins arrive: how often is the model right? |
| ~10 min | 4 · Choosing k: overfitting and underfitting (SLO 06A) |
| ~5 min | 5 · Judged by your neighbors |
| ~3 min | 6 · Zooming out |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk

<sub>Artwork by @allison_horst · Data: Gorman, Williams & Fraser (2014), Palmer Station LTER, via the palmerpenguins package</sub>

---
## 1 · Look at the Data · ~5 min

**penguins** — one row per penguin, 2007–2009:

* `bill_length_mm`, `bill_depth_mm` — two measurements of its bill
* `species` — **Adelie**, **Chinstrap** or **Gentoo**, confirmed by the field team

It's the end of **2008**: the team has two seasons of penguins. 2009 hasn't happened yet.

In [ ]:
import pandas as pd
import plotly.express as px

penguins = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/penguins.csv")
penguins = penguins.dropna(subset=["bill_length_mm", "bill_depth_mm"])   # 2 penguins have no bill measurements

known = penguins[penguins["year"] <= 2008]                              # the 2007 and 2008 seasons
print(known.shape)
known[["species", "island", "bill_length_mm", "bill_depth_mm", "year"]].head()

In [ ]:
COLORS = {"Adelie": "#FF8C00", "Chinstrap": "#A034F0", "Gentoo": "#159090"}
SYMBOLS = {"Adelie": "circle", "Chinstrap": "triangle-up", "Gentoo": "square"}

fig = px.scatter(known, x="bill_length_mm", y="bill_depth_mm", color="species", symbol="species",
                 color_discrete_map=COLORS, symbol_map=SYMBOLS, title="223 Penguins, 2007–2008")
fig.show()

> 💬 **Question:** Last week's listeners came with no labels. What's different here?

<details><summary>Answer</summary>

Every penguin has a **label** — its species. Each species has its own region, with some overlap at the borders.
</details>

---
## 2 · Ask the Neighbors: k-Nearest Neighbors (SLO 06A) · ~10 min

Three penguins measured in **2009** (★). Nobody has told the model their species.

In [ ]:
mystery = pd.DataFrame({"bill_length_mm": [49.1, 44.1, 50.8],
                        "bill_depth_mm": [14.5, 18.0, 18.5]}, index=["A", "B", "C"])

fig = px.scatter(known, x="bill_length_mm", y="bill_depth_mm", color="species", symbol="species",
                 color_discrete_map=COLORS, symbol_map=SYMBOLS, title="Three New Penguins")
fig.add_scatter(x=mystery["bill_length_mm"], y=mystery["bill_depth_mm"], text=mystery.index,
                mode="markers+text", textposition="top center", name="new penguin",
                marker=dict(symbol="star", size=17, color="black"))
fig.show()

> 📄 **Handout 1 — Which species?** Guess A, B and C.
>
> 📄 **Handout 2 — Ask the neighbors.** Zoomed in on **B**: circle its **nearest** neighbor, then circle its **5** nearest. What does each group vote? 🗣️ Compare with a neighbor.

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/06/images/knn_animation.gif" width="440" alt="Animation: a new black point among green and red points; a dashed circle grows around it until it holds its 5 nearest neighbors, and the new point takes the majority's color.">

That recipe is **k-nearest neighbors (kNN)**:

1. **Measure** the distance from the new penguin to every known penguin
2. **Take** the `k` closest
3. **Vote** — the most common species wins

`k` is a **hyperparameter**: we choose it; the data doesn't.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_known = known[["bill_length_mm", "bill_depth_mm"]]      # features
y_known = known["species"]                                # label

knn1 = KNeighborsClassifier(n_neighbors=1).fit(X_known, y_known)
knn5 = KNeighborsClassifier(n_neighbors=5).fit(X_known, y_known)

X_mystery = mystery[["bill_length_mm", "bill_depth_mm"]]
mystery["k = 1 says"] = knn1.predict(X_mystery)
mystery["k = 5 says"] = knn5.predict(X_mystery)
mystery

In [ ]:
distances, rows = knn5.kneighbors(X_mystery.loc[["B"]])      # B's 5 nearest known penguins
b_neighbors = known.iloc[rows[0]][["species", "bill_length_mm", "bill_depth_mm"]]
b_neighbors["distance_mm"] = distances[0].round(2)
b_neighbors

> 💬 **Question:** k = 1 and k = 5 disagree about B. The field team says B is an **Adelie**. What went wrong with k = 1?

<details><summary>Answer</summary>

* B's single nearest neighbor is a **Gentoo** that happens to sit close by — k = 1 copies it
* Its 5 nearest: 3 Adelie, 1 Gentoo, 1 Chinstrap → **Adelie** wins
* A (Gentoo) and C (Chinstrap): both k's agree, and both are right
</details>

---
## 3 · New Penguins Arrive: How Often Is the Model Right? · ~7 min

**2009:** the team measures **119 new penguins** and confirms each one's species. The model learned only from 2007–2008.

**Accuracy** = the share of penguins whose species the model gets right.

In [ ]:
from sklearn.metrics import accuracy_score

new = penguins[penguins["year"] == 2009]                  # the new season
X_new, y_new = new[["bill_length_mm", "bill_depth_mm"]], new["species"]

print("k = 5 on the new penguins:", round(accuracy_score(y_new, knn5.predict(X_new)), 3))

> 🎯 **Predict First:** Now **k = 1**, scored on the **223 penguins it learned from**. What accuracy?

In [ ]:
for name, model in [("k = 1", knn1), ("k = 5", knn5)]:
    print(name, "| known penguins:", round(accuracy_score(y_known, model.predict(X_known)), 3),
          "| new penguins:", round(accuracy_score(y_new, model.predict(X_new)), 3))

<details><summary>Answer</summary>

**100%** — every known penguin's nearest neighbor is *itself*. That's **memorizing**, not learning.

| | known penguins (2007–08) | new penguins (2009) |
|---|---|---|
| k = 1 | **100%** | 91.6% |
| k = 5 | 98.2% | **94.1%** |

A score on penguins the model has already seen says little about the next ones.
</details>

---
## 4 · Choosing k: Overfitting and Underfitting (SLO 06A) · ~10 min

Color every spot of the plot by the species the model would predict there: the model's **map**.

> 🎯 **Predict First:** What does the map look like with k = 1? With k = 100?

In [ ]:
# @title Helper: draw kNN's map for several k (double-click to see the code)
import numpy as np
import matplotlib.pyplot as plt

def plot_maps(ks, X, y):
    """One panel per k: the background is what kNN predicts at every spot; the dots are the penguins it learned from."""
    species = list(COLORS)                                               # Adelie, Chinstrap, Gentoo
    xx, yy = np.meshgrid(np.linspace(31, 61, 300), np.linspace(12.5, 22, 200))
    grid = pd.DataFrame({"bill_length_mm": xx.ravel(), "bill_depth_mm": yy.ravel()})
    fig, axes = plt.subplots(1, len(ks), figsize=(5 * len(ks), 4), sharey=True)
    for ax, k in zip(axes, ks):
        model = KNeighborsClassifier(n_neighbors=k).fit(X, y)
        z = pd.Series(model.predict(grid)).map(species.index).to_numpy().reshape(xx.shape)
        ax.contourf(xx, yy, z, levels=[-0.5, 0.5, 1.5, 2.5], colors=list(COLORS.values()), alpha=0.25)
        for s, marker in zip(species, ["o", "^", "s"]):
            ax.scatter(X.loc[y == s, "bill_length_mm"], X.loc[y == s, "bill_depth_mm"],
                       s=14, c=COLORS[s], marker=marker, label=s)
        ax.set_title(f"k = {k}")
        ax.set_xlabel("bill_length_mm")
    axes[0].set_ylabel("bill_depth_mm")
    axes[0].legend(loc="upper left", fontsize=8)
    plt.show()

In [ ]:
plot_maps([1, 5, 100], X_known, y_known)

> 📄 **Handout 3 — Three maps.** Circle an "island" in the k = 1 map. Where did the Chinstraps go at k = 100? Under each map, write *too wiggly*, *about right* or *too simple*. 🗣️ Compare with a neighbor.

<details><summary>Answer</summary>

* **k = 1 — too wiggly:** every odd penguin gets its own island. The map follows the *noise* → **overfitting**
* **k = 100 — too simple:** the big species outvote everyone; the Chinstrap region nearly vanishes → **underfitting**
* **k = 5 — in between**
</details>

Which k is best? Score every k on both seasons.

> 📄 **Handout 4 — Which k?** Circle the best k for the new penguins. Label the overfitting zone and the underfitting zone.

In [ ]:
scores = []
for k in range(1, 150, 2):
    model = KNeighborsClassifier(n_neighbors=k).fit(X_known, y_known)
    scores.append({"k": k,
                   "known penguins (2007–08)": accuracy_score(y_known, model.predict(X_known)),
                   "new penguins (2009)": accuracy_score(y_new, model.predict(X_new))})
scores = pd.DataFrame(scores).melt(id_vars="k", var_name="scored on", value_name="accuracy")   # long format, as in Week 4

fig = px.line(scores, x="k", y="accuracy", color="scored on", line_dash="scored on", markers=True,
              title="kNN Accuracy vs. k")
fig.show()

<details><summary>Answer</summary>

* **Best for the new penguins: k = 3 or 5** (94.1%)
* **Small k:** near-perfect on known penguins, worse on new ones → **overfitting**
* **Large k:** poor on both — from k = 95 on, not one new penguin is called a Chinstrap → **underfitting**
</details>

### The idea of the semester

| | on data it learned from | on new data |
|---|---|---|
| **Overfitting** — too flexible | great | worse |
| **Underfitting** — too simple | poor | poor |
| **Good fit** | good | good |

`k` is kNN's knob. Most models this semester have one.

---
## 5 · Judged by Your Neighbors · ~5 min

kNN makes one assumption: **you are like your neighbors.**

> "…when left to their own devices people have a tendency to associate with others who are similar to them in various salient ways: birds of a feather flock together. … A recommendation algorithm might find each person's “nearest neighbors” in terms of positions in an abstract space defined by patterns of consumption or purchasing or voting or any such quality."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** At k = 100, the Chinstraps — the smallest group — were outvoted almost everywhere. If the dots were *people* and the label were "will repay a loan", who could be outvoted? And who decides what "neighbor" means?

<details><summary>Answer</summary>

* **Large k:** the majority around you decides — small groups get outvoted
* **k = 1:** a single look-alike decides for you
* **"Neighbor" is a human choice:** which columns (zip code? purchases?) and what counts as close
* Harmless for penguins. For loans, jobs or policing, your neighbors become your prediction
</details>

---
## 6 · Zooming Out · ~3 min

**Features + labels** → **supervised learning**: learn from examples that come with answers, then predict the answers for new cases.

* Label is a **category** (species, spam or not) → **classification** — today
* Label is a **number** (body mass, price) → **regression** — Week 10

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/06/images/fit_predict.png" width="560" alt="Diagram: training data (X and y) and a model object go into fit, giving a fitted model; the fitted model and new data X go into predict, giving predicted y's.">

Every supervised model in scikit-learn works the same way: `model.fit(X, y)`, then `model.predict(X_new)`.

**Wednesday:** today, 2009's penguins told us how the model does on new data. Usually there's no next season yet — so how do we test a model? And is accuracy the only number that matters? *(SLOs 06B, 06C)*